# 05 — Feature engineering

Tạo đặc trưng từ kết luận của EDA (notebook 04). Mỗi nhóm dưới đây gắn với một kết quả đã đo được,
hoặc ghi rõ là **giả thuyết** nếu EDA chưa kiểm tra.

| Căn cứ | Feature tạo ra | Mục EDA |
|---|---|---|
| Chu kỳ ngày, tuần, mùa rõ | `hour_sin/cos`, `dow_sin/cos`, `month_sin/cos` | VI |
| Tốc độ gió chi phối, lặng gió làm bụi tích tụ | `wind_calm`, `wind_roll6` | 10.3, 10.7 |
| Hướng gió khác biệt nhỏ, giữ để bước chọn quyết định | `wind_sin/cos` | 10.6 |
| PM10 gần như đồng biến với PM2.5 (Spearman 0.95) | PM10 trực tiếp, `PM10_PM25_ratio` | IX |
| *Giả thuyết:* áp suất tăng nhanh báo hiệu thời tiết ổn định | `Pressure_anom_24` | kiểm tra ở 10.4b |
| Chỉ mưa lớn mới rửa được bụi | `hours_since_rain`, `rain_24h_sum`, `Accumulated Hours of Rain` | 10.5 |
| *Giả thuyết:* ngày lễ giảm giao thông | `IsHoliday` | kiểm tra ở 6.3 |
| Độ ẩm đất, lớp biên, cháy có tín hiệu sau khi loại mùa | 5 cột dữ liệu ngoài | 13.1 |
| PM2.5 phụ thuộc lịch sử của chính nó | 32 feature lag / rolling (chỉ cho mô hình dạng bảng) | VII |




In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data' / 'processed'
RESULT_DIR = ROOT / 'results'

# Nạp module trong src/ trực tiếp (không qua package src)
sys.path.insert(0, str(ROOT / 'src'))

pd.set_option('display.width', 160)

In [2]:
VN_HOLIDAYS = frozenset({
    # Tết Dương lịch (kèm ngày nghỉ bù khi 1/1 rơi vào cuối tuần)
    '2022-01-01', '2022-01-02', '2022-01-03',
    '2023-01-01', '2023-01-02',
    '2024-01-01',
    '2025-01-01',
    '2026-01-01',
    # Giỗ Tổ Hùng Vương (10/3 âm lịch)
    '2022-04-10', '2022-04-11',
    '2023-04-29',
    '2024-04-18',
    '2025-04-07',
    '2026-04-26',
    # 30/4 - 1/5 (kèm nghỉ bù)
    '2022-04-30', '2022-05-01', '2022-05-02', '2022-05-03',
    '2023-04-30', '2023-05-01', '2023-05-02', '2023-05-03',
    '2024-04-30', '2024-05-01',
    '2025-04-30', '2025-05-01',
    '2026-04-30', '2026-05-01',
    # Quốc khánh 2/9
    '2022-09-01', '2022-09-02', '2022-09-03', '2022-09-04',
    '2023-09-01', '2023-09-02', '2023-09-03', '2023-09-04',
    '2024-08-31', '2024-09-01', '2024-09-02', '2024-09-03',
    '2025-08-30', '2025-08-31', '2025-09-01', '2025-09-02',
    '2026-08-31', '2026-09-01', '2026-09-02',
    # Tết Nguyên Đán
    '2022-01-29', '2022-01-30', '2022-01-31', '2022-02-01', '2022-02-02',
    '2022-02-03', '2022-02-04', '2022-02-05', '2022-02-06',
    '2023-01-20', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24',
    '2023-01-25', '2023-01-26',
    '2024-02-08', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12',
    '2024-02-13', '2024-02-14',
    '2025-01-25', '2025-01-26', '2025-01-27', '2025-01-28', '2025-01-29',
    '2025-01-30', '2025-01-31', '2025-02-01', '2025-02-02',
    '2026-02-14', '2026-02-15', '2026-02-16', '2026-02-17', '2026-02-18',
    '2026-02-19', '2026-02-20', '2026-02-21', '2026-02-22',
})

## I. Đọc dữ liệu và kiểm tra lưới giờ

In [3]:
df = pd.read_csv(DATA_DIR / 'merged_data.csv', parse_dates=['Local Time'])
df = df.set_index('Local Time').sort_index()

assert not df.index.duplicated().any(), 'Có timestamp trùng'

# Ép về lưới giờ liên tục: giờ nào không có dòng sẽ hiện ra dưới dạng NaN thay vì biến mất
n_before = len(df)
df = df.asfreq('h')
print('Số giờ thêm vào để lấp lưới:', len(df) - n_before)

# Bỏ AQI và các cột EDA kết luận không dùng.
# Boundary Layer Height: nguồn thiếu trọn 01-06/2024, giữ lại sẽ làm mất 6 tháng dữ liệu train
# CO: median tăng gấp 4 lần 2022-2026, do nguồn dữ liệu đổi chứ không phải thực tế
drop_cols = [c for c in ['AQI', 'UV Index', 'Solar Radiation', 'Dew Point', 'Fire Dist Min',
                         'Boundary Layer Height', 'CO'] if c in df.columns]
df = df.drop(columns=drop_cols)
print('Đã bỏ:', drop_cols)

print('Kích thước:', df.shape)
print('Từ', df.index.min(), 'đến', df.index.max())
display(df.head())

Số giờ thêm vào để lấp lưới: 0
Đã bỏ: ['AQI', 'UV Index', 'Solar Radiation', 'Dew Point', 'Fire Dist Min', 'Boundary Layer Height', 'CO']
Kích thước: (40608, 16)
Từ 2022-01-13 00:00:00 đến 2026-08-31 23:00:00


,NO2,O3,PM10,PM25,SO2,Temperature,Relative Humidity,Precipitation,Clouds,Pressure,Wind Speed,Wind Direction,Soil Moisture,Wind Speed 10m,Fire Count 100km 3d,Fire FRP 100km 7d
Local Time,,,,,,,,,,,,,,,,
2022-01-13 00:00:00,16.7,86.7,109.1,65.7,54.7,17.0,93.0,0.0,100.0,1019.0,1.6,106.0,0.277,1.69,NaN,NaN
2022-01-13 01:00:00,17.0,87.0,110.8,66.7,59.0,16.8,97.0,0.0,100.0,1019.0,1.2,98.0,0.277,1.67,NaN,NaN
2022-01-13 02:00:00,16.0,87.3,107.1,63.7,58.7,16.4,97.0,0.0,91.0,1019.0,1.2,78.0,0.278,1.67,NaN,NaN
2022-01-13 03:00:00,15.0,87.7,103.5,61.6,58.3,16.1,97.0,0.0,83.0,1019.0,1.2,58.0,0.278,1.83,NaN,NaN
2022-01-13 04:00:00,14.0,88.0,99.8,59.6,58.0,15.7,97.0,0.0,75.0,1018.0,1.6,46.0,0.278,2.09,NaN,NaN


## II. Cố định mốc chia dữ liệu

In [4]:
# Chia theo năm để Validation và Test đều có trọn một chu kỳ mùa (có cả mùa đông)
train_end = pd.Timestamp('2024-08-31 23:00')
val_start, val_end = pd.Timestamp('2024-09-01 00:00'), pd.Timestamp('2025-08-31 23:00')
test_start = pd.Timestamp('2025-09-01 00:00')

split_table = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Start': [df.index[0], val_start, test_start],
    'End': [train_end, val_end, df.index[-1]]
})
split_table['Số giờ'] = [len(df.loc[s:e]) for s, e in zip(split_table['Start'], split_table['End'])]
split_table['PM2.5 TB'] = [round(df.loc[s:e, 'PM25'].mean(), 1)
                           for s, e in zip(split_table['Start'], split_table['End'])]
display(split_table)
split_table[['Split', 'Start', 'End']].to_csv(DATA_DIR / 'split_dates.csv', index=False)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
split_table[['Split', 'Start', 'End']].to_csv(RESULT_DIR / 'split_dates.csv', index=False)

is_train = df.index <= train_end

print('Số giờ PM2.5 thiếu:', int(df['PM25'].isna().sum()))
print('Số giờ PM2.5 >= PM10 (nghi ngờ):', int(((df['PM25'] >= df['PM10']) & df['PM10'].notna()).sum()))

# Giữ bản thô (trước khi thêm feature) để mục XII chạy lại toàn bộ hàm tạo feature khi kiểm tra rò rỉ
raw_inputs = df.copy()

,Split,Start,End,Số giờ,PM2.5 TB
0,Train,2022-01-13,2024-08-31 23:00:00,23088,40.9
1,Validation,2024-09-01,2025-08-31 23:00:00,8760,51.1
2,Test,2025-09-01,2026-08-31 23:00:00,8760,41.9


Số giờ PM2.5 thiếu: 0
Số giờ PM2.5 >= PM10 (nghi ngờ): 41


## III. Đặc trưng thời gian

In [5]:
def time_features(data):
    idx = data.index
    return pd.DataFrame({
        'hour_sin': np.sin(2 * np.pi * idx.hour / 24),
        'hour_cos': np.cos(2 * np.pi * idx.hour / 24),
        'dow_sin': np.sin(2 * np.pi * idx.dayofweek / 7),
        'dow_cos': np.cos(2 * np.pi * idx.dayofweek / 7),
        'month_sin': np.sin(2 * np.pi * (idx.month - 1) / 12),
        'month_cos': np.cos(2 * np.pi * (idx.month - 1) / 12),
    }, index=idx)

df[list(time_features(df).columns)] = time_features(df)
display(df[['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']].head())

,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
Local Time,,,,,,
2022-01-13 00:00:00,0.000000,1.000000,0.433884,-0.900969,0.0,1.0
2022-01-13 01:00:00,0.258819,0.965926,0.433884,-0.900969,0.0,1.0
2022-01-13 02:00:00,0.500000,0.866025,0.433884,-0.900969,0.0,1.0
2022-01-13 03:00:00,0.707107,0.707107,0.433884,-0.900969,0.0,1.0
2022-01-13 04:00:00,0.866025,0.500000,0.433884,-0.900969,0.0,1.0


## IV. Gió


In [6]:
WIND_CALM_THRESHOLD = 1.6  # m/s, ranh giới nhóm 2 và 3 ở EDA mục 10.3

def wind_features(data):
    wind_rad = np.deg2rad(data['Wind Direction'])
    calm = (data['Wind Speed'] <= WIND_CALM_THRESHOLD).astype(float)
    calm[data['Wind Speed'].isna()] = np.nan
    return pd.DataFrame({
        'wind_sin': np.sin(wind_rad),
        'wind_cos': np.cos(wind_rad),
        'wind_calm': calm,
        # Gió trung bình 6 giờ trước, không gồm giờ hiện tại
        'wind_roll6': data['Wind Speed'].shift(1).rolling(6).mean(),
    }, index=data.index)

new = wind_features(df)
df[list(new.columns)] = new

print(f"Tỷ lệ giờ lặng gió: {df['wind_calm'].mean() * 100:.1f}%")
display(df[['Wind Direction', 'wind_sin', 'wind_cos', 'wind_calm', 'wind_roll6']].head(8))

Tỷ lệ giờ lặng gió: 41.3%


,Wind Direction,wind_sin,wind_cos,wind_calm,wind_roll6
Local Time,,,,,
2022-01-13 00:00:00,106.0,0.961262,-0.275637,1.0,NaN
2022-01-13 01:00:00,98.0,0.990268,-0.139173,1.0,NaN
2022-01-13 02:00:00,78.0,0.978148,0.207912,1.0,NaN
2022-01-13 03:00:00,58.0,0.848048,0.529919,1.0,NaN
2022-01-13 04:00:00,46.0,0.719340,0.694658,1.0,NaN
2022-01-13 05:00:00,46.0,0.719340,0.694658,0.0,NaN
2022-01-13 06:00:00,42.0,0.669131,0.743145,0.0,1.466667
2022-01-13 07:00:00,320.0,-0.642788,0.766044,0.0,1.600000


## V. Tỷ lệ bụi thô và bụi mịn

In [7]:
def raw_ratio(data):
    return data['PM10'] / data['PM25'].clip(lower=1.0)

# Ngưỡng cắt tính một lần trên Train, sau đó cố định
ratio = raw_ratio(df)
RATIO_LOW, RATIO_HIGH = ratio[is_train].quantile([0.01, 0.99])

def ratio_feature(data):
    return pd.DataFrame({'PM10_PM25_ratio': raw_ratio(data).clip(RATIO_LOW, RATIO_HIGH)}, index=data.index)

df['PM10_PM25_ratio'] = ratio_feature(df)['PM10_PM25_ratio']

print(f'Ngưỡng cắt (từ Train): {RATIO_LOW:.3f} đến {RATIO_HIGH:.3f}')
print('Số giờ bị cắt:', int(((ratio < RATIO_LOW) | (ratio > RATIO_HIGH)).sum()))
display(pd.DataFrame({'Trước cắt': ratio.describe(), 'Sau cắt': df['PM10_PM25_ratio'].describe()}).round(3))

Ngưỡng cắt (từ Train): 1.024 đến 3.253
Số giờ bị cắt: 827


,Trước cắt,Sau cắt
count,40608.000,40608.000
mean,1.583,1.556
std,0.691,0.318
min,0.998,1.024
25%,1.457,1.457
50%,1.501,1.501
75%,1.614,1.614
max,39.667,3.253


## VI. Áp suất đổi nhanh (giả thuyết)

In [8]:
def pressure_feature(data):
    pressure_prev = data['Pressure'].shift(1)
    return pd.DataFrame({'Pressure_anom_24': pressure_prev - pressure_prev.rolling(24).mean()}, index=data.index)

df['Pressure_anom_24'] = pressure_feature(df)['Pressure_anom_24']
display(df[['Pressure', 'Pressure_anom_24']].describe().round(3))

,Pressure,Pressure_anom_24
count,40608.000,40584.000
mean,1009.654,-0.005
std,7.280,1.773
min,983.000,-9.333
25%,1004.000,-1.125
50%,1009.000,0.000
75%,1015.000,1.083
max,1033.000,10.000


## VII. Mưa và hiệu ứng rửa bụi

In [9]:
def rain_features(data):
    precip = data['Precipitation'].fillna(0)
    raining = (precip > 0).astype(int)
    groups = (raining != raining.shift()).cumsum()

    out = pd.DataFrame(index=data.index)
    out['Accumulated Hours of Rain'] = raining.groupby(groups).cumsum().astype(float)

    # Đếm từ 1 trong đoạn khô, đang mưa thì bằng 0
    dry_count = raining.groupby(groups).cumcount() + 1
    out['hours_since_rain'] = dry_count.where(raining == 0, 0).astype(float)

    # Đoạn khô đầu chuỗi chưa có trận mưa nào trước đó
    first_rain = raining.idxmax() if raining.any() else data.index[-1]
    out.loc[out.index < first_rain, 'hours_since_rain'] = np.nan

    out['rain_24h_sum'] = precip.rolling(24).sum()
    return out

new = rain_features(df)
df[list(new.columns)] = new

display(df[['Precipitation', 'Accumulated Hours of Rain', 'hours_since_rain', 'rain_24h_sum']]
        .loc['2025-09-01':'2025-09-03'].head(30))

,Precipitation,Accumulated Hours of Rain,hours_since_rain,rain_24h_sum
Local Time,,,,
2025-09-01 00:00:00,0.25,4.0,0.0,7.75
2025-09-01 01:00:00,0.00,0.0,1.0,7.50
2025-09-01 02:00:00,0.00,0.0,2.0,7.50
2025-09-01 03:00:00,0.00,0.0,3.0,7.50
2025-09-01 04:00:00,0.00,0.0,4.0,7.50
2025-09-01 05:00:00,0.25,1.0,0.0,7.75
2025-09-01 06:00:00,0.00,0.0,1.0,7.75
2025-09-01 07:00:00,0.00,0.0,2.0,7.75
2025-09-01 08:00:00,0.25,1.0,0.0,8.00


## VIII. Ngày lễ Việt Nam (giả thuyết)

In [10]:
def holiday_feature(data):
    flag = data.index.strftime('%Y-%m-%d').isin(list(VN_HOLIDAYS)).astype(int)
    return pd.DataFrame({'IsHoliday': flag}, index=data.index)

df['IsHoliday'] = holiday_feature(df)['IsHoliday']

print('Số giờ rơi vào ngày lễ:', int(df['IsHoliday'].sum()))
print('Số ngày lễ có trong dữ liệu:', df.loc[df['IsHoliday'] == 1].index.normalize().nunique())

Số giờ rơi vào ngày lễ: 1992
Số ngày lễ có trong dữ liệu: 83


## IX. Lấp chỗ trống ngắn của dữ liệu ngoài

In [11]:
EXTERNAL_COLS = [c for c in ['Soil Moisture', 'Boundary Layer Height', 'Wind Speed 10m',
                             'Fire Count 100km 3d', 'Fire FRP 100km 7d'] if c in df.columns]

def external_features(data):
    return data[EXTERNAL_COLS].ffill(limit=3)

before = df[EXTERNAL_COLS].isna().sum()
df[EXTERNAL_COLS] = external_features(df)
after = df[EXTERNAL_COLS].isna().sum()

external_numeric = EXTERNAL_COLS
print(pd.DataFrame({'Trước': before, 'Sau': after, 'Đã lấp': before - after}))

                     Trước  Sau  Đã lấp
Soil Moisture            0    0       0
Wind Speed 10m           0    0       0
Fire Count 100km 3d     24   24       0
Fire FRP 100km 7d       24   24       0


## X. Feature lịch sử PM2.5 (chỉ cho mô hình dạng bảng)

In [12]:
LAGS = [1, 2, 3, 6, 8, 12, 24, 36, 48, 72, 96, 120, 144, 168]

def pm25_history_features(data):
    pm25 = data['PM25']
    history = {}
    for lag in LAGS:
        history[f'PM25_lag_{lag}'] = pm25.shift(lag)

    shifted = pm25.shift(1)
    for w in (6, 12, 24, 72):
        history[f'PM25_roll_mean_{w}'] = shifted.rolling(w).mean()
        history[f'PM25_roll_std_{w}'] = shifted.rolling(w).std()
    for w in (6, 24):
        history[f'PM25_roll_max_{w}'] = shifted.rolling(w).max()
        history[f'PM25_roll_min_{w}'] = shifted.rolling(w).min()

    history['PM25_diff_1'] = shifted.diff(1)
    history['PM25_diff_24'] = shifted.diff(24)

    history = pd.DataFrame(history, index=data.index)

    same_3d = [f'PM25_lag_{d}' for d in (24, 48, 72)]
    same_7d = [f'PM25_lag_{d}' for d in (24, 48, 72, 96, 120, 144, 168)]
    history['PM25_same_hour_mean_3d'] = history[same_3d].mean(axis=1, skipna=False)
    history['PM25_same_hour_mean_7d'] = history[same_7d].mean(axis=1, skipna=False)
    history['PM25_same_hour_std_7d'] = history[same_7d].std(axis=1, skipna=False)
    history['PM25_same_hour_max_7d'] = history[same_7d].max(axis=1, skipna=False)
    return history

history = pm25_history_features(df)
# Ghép một lần để tránh cảnh báo DataFrame bị phân mảnh
df = pd.concat([df, history], axis=1)
pm25_history = list(history.columns)

print('Số feature lịch sử PM2.5:', len(pm25_history))
display(df[pm25_history].tail(3).round(2))

Số feature lịch sử PM2.5: 32


,PM25_lag_1,PM25_lag_2,PM25_lag_3,PM25_lag_6,PM25_lag_8,PM25_lag_12,PM25_lag_24,PM25_lag_36,PM25_lag_48,PM25_lag_72,...,PM25_roll_max_6,PM25_roll_min_6,PM25_roll_max_24,PM25_roll_min_24,PM25_diff_1,PM25_diff_24,PM25_same_hour_mean_3d,PM25_same_hour_mean_7d,PM25_same_hour_std_7d,PM25_same_hour_max_7d
Local Time,,,,,,,,,,,,,,,,,,,,,
2026-08-31 21:00:00,27.0,28.0,18.0,21.0,18.0,35.0,36.0,26.0,17.0,17.0,...,28.0,12.0,61.0,12.0,-1.0,-22.0,23.33,26.00,10.49,39.0
2026-08-31 22:00:00,35.0,27.0,28.0,12.0,18.0,26.0,52.0,21.0,21.0,20.0,...,35.0,12.0,61.0,12.0,8.0,-1.0,31.00,29.43,13.10,52.0
2026-08-31 23:00:00,38.0,35.0,27.0,21.0,21.0,24.0,56.0,21.0,33.0,22.0,...,38.0,18.0,61.0,12.0,3.0,-14.0,37.00,35.14,13.42,56.0


## XI. Danh sách đặc trưng theo nhóm

In [13]:
feature_groups = {
    'target_history': ['PM25'],
    'pollutants': ['PM10', 'NO2', 'SO2', 'O3'],
    'weather_base': ['Temperature', 'Relative Humidity', 'Precipitation', 'Clouds',
                     'Pressure', 'Wind Speed'],
    'weather_derived': ['wind_sin', 'wind_cos', 'wind_calm', 'wind_roll6', 'Pressure_anom_24'],
    'external': external_numeric,
    'derived': ['PM10_PM25_ratio', 'hours_since_rain', 'rain_24h_sum',
                'Accumulated Hours of Rain', 'IsHoliday'],
    'time': ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos'],
    'pm25_history_tabular_only': pm25_history,
}

# Cột khí tượng nào không có trong file gốc thì bỏ khỏi nhóm và báo ra
for group, cols in feature_groups.items():
    absent = [c for c in cols if c not in df.columns]
    if absent:
        print(f'Nhóm {group}: không có cột {absent}, bỏ khỏi danh sách')
        feature_groups[group] = [c for c in cols if c in df.columns]

input_columns = [c for cols in feature_groups.values() for c in cols]
assert len(input_columns) == len(set(input_columns)), 'Có cột bị trùng giữa các nhóm'

for group, cols in feature_groups.items():
    print(f'  {group:<28} {len(cols):>2} cột')
print('Tổng feature:', len(input_columns))

  target_history                1 cột
  pollutants                    4 cột
  weather_base                  6 cột
  weather_derived               5 cột
  external                      4 cột
  derived                       5 cột
  time                          6 cột
  pm25_history_tabular_only    32 cột
Tổng feature: 63


## XII. Lưu và kiểm tra

In [14]:

data_features = df[input_columns].copy()

# 1. Lưới giờ liên tục
step = data_features.index.to_series().diff().dropna()
assert (step == pd.Timedelta(hours=1)).all(), 'Lưới giờ bị đứt'
assert data_features.index.is_unique

# 2. Kiểm tra rò rỉ: chạy lại toàn bộ các hàm tạo feature trên dữ liệu bị làm nhiễu từ mốc probe trở đi
FEATURE_BUILDERS = [time_features, wind_features, ratio_feature, pressure_feature,
                    rain_features, holiday_feature, external_features, pm25_history_features]

def build_all(raw):
    return pd.concat([builder(raw) for builder in FEATURE_BUILDERS], axis=1)

def perturb_after(raw, probe, seed=0):
    rng = np.random.default_rng(seed)
    out = raw.copy()
    after = out.index >= probe
    for col in out.select_dtypes('number').columns:
        noise = rng.normal(0, out[col].std() + 1, after.sum())
        out.loc[after, col] = out.loc[after, col] + noise + 1000
    return out

reference = build_all(raw_inputs)
assert np.allclose(reference.to_numpy(float), df[reference.columns].to_numpy(float), equal_nan=True), \
    'Feature trong df khác với feature tính lại từ bản thô'

for probe in [df.index[len(df) // 3], df.index[len(df) // 2], df.index[int(len(df) * 0.85)]]:
    perturbed = build_all(perturb_after(raw_inputs, probe))
    before_probe = reference.index < probe
    changed = [c for c in reference.columns
               if not np.allclose(reference.loc[before_probe, c].to_numpy(float),
                                  perturbed.loc[before_probe, c].to_numpy(float), equal_nan=True)]
    assert not changed, f'Các feature dùng giá trị tương lai (mốc {probe}): {changed}'
print('Kiểm tra rò rỉ: đạt cho', len(reference.columns), 'feature, 3 mốc')

# 3. Số giờ đầy đủ
seq_cols = [c for c in input_columns if c not in pm25_history]
full_seq = data_features[seq_cols].notna().all(axis=1)
full_all = data_features[input_columns].notna().all(axis=1)
print(f'Tổng số giờ: {len(data_features)}')
print(f'Giờ đủ feature cho GRU/LSTM : {int(full_seq.sum())} ({full_seq.mean() * 100:.1f}%)')
print(f'Giờ đủ cả lịch sử PM2.5     : {int(full_all.sum())} ({full_all.mean() * 100:.1f}%)')

na_left = data_features[seq_cols].isna().sum()
print('Cột còn NaN (không tính lịch sử PM2.5):')
print(na_left[na_left > 0].to_string() if (na_left > 0).any() else '  không có')

data_features = data_features.reset_index()
data_features.to_csv(DATA_DIR / 'feature_data.csv', index=False, encoding='utf-8-sig')



print('Đã lưu feature_data.csv:', data_features.shape)


Kiểm tra rò rỉ: đạt cho 52 feature, 3 mốc
Tổng số giờ: 40608
Giờ đủ feature cho GRU/LSTM : 40584 (99.9%)
Giờ đủ cả lịch sử PM2.5     : 40440 (99.6%)
Cột còn NaN (không tính lịch sử PM2.5):
wind_roll6              6
Pressure_anom_24       24
Fire Count 100km 3d    24
Fire FRP 100km 7d      24
hours_since_rain        9
rain_24h_sum           23
Đã lưu feature_data.csv: (40608, 64)
